# ETL walkthrough: from the downloaded files to the database

This notebook runs the data pipeline in `etl/` and shows what happens at every step: each stage is explained, run, and
its result displayed.

**What the pipeline does.** `download_data.py` has already saved each dataset under `raw/`. For each dataset the ETL
(1) checks the file against its checksum, (2) unzips it, (3) cleans it into a consistent shape, (4) checks the result,
and (5) optionally loads it into PostGIS as schema `vibrancy`. It only writes `staging/` and that schema, never any other table in the database.

**Before you start:** run `python download_data.py` (see its docstring; in Docker, the command is in `etl/README.md`).
A source that hasn't been downloaded is reported as `NOT RUN`, and the others carry on.

**How to run this notebook**

- **In VS Code:** choose a kernel that has the libraries: Python 3.11 with `etl/requirements.txt` installed (for
  example a conda environment called `vibrancy`). Then run the cells from the top.
- **In Docker, with nothing to install** (runs every cell and saves the result with its outputs in `output/`):

  ```bash
  docker compose run --rm --no-deps --user "$(id -u):$(id -g)" -e HOME=/tmp --entrypoint jupyter etl \
      nbconvert --to notebook --execute etl_walkthrough.ipynb --output-dir output \
      --output etl_walkthrough.executed.ipynb --ExecutePreprocessor.kernel_name=python3 --ExecutePreprocessor.timeout=-1
  ```

**What it costs.** The pipeline reads about 750 MB of downloads and unzips them to about 2.4 GB, which takes a few
minutes. To try it on a few small sources first, set `SOURCES` in the next cell.

## 0. Settings

Everything you might want to change is in the next cell.

| Setting | Meaning |
|---|---|
| `SOURCES` | Which sources to run. `None` is all of them. Or a list such as `["abs_population", "hospitals"]`. |
| `LOAD_TO` | `None`: files only, nothing goes into a database (the safe default). `"pgadmin"`: your PostgreSQL on port 5432 (the one pgAdmin shows), using `Credentials.json`, database `postgres`. `"compose"`: the database docker compose starts (only works inside Docker). |
| `RESET_FILES` | `True` deletes `raw/` and `staging/` in the last cell. It never touches any database. |

In [ ]:
import os

# Which sources to run: None = every source, or a list such as ["abs_population", "hospitals"].
# The environment variable ETL_SOURCES="abs_population hospitals" does the same (that is how the tests run this notebook).
SOURCES = os.environ.get("ETL_SOURCES", "").split() or None

LOAD_TO = None         # None = files only;  "pgadmin" = your PostgreSQL on port 5432;  "compose" = docker compose's database
RESET_FILES = False    # True = delete raw/ and staging/ in the last cell (never a database)

## 1. Where this is running

In [ ]:
import inspect, json, platform, shutil, sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "etl").is_dir():
    raise SystemExit("Open this notebook from the project folder (the one that contains etl/).")
sys.path.insert(0, str(ROOT))
IN_DOCKER = Path("/.dockerenv").exists()

from etl import cli, core, load, quality, schemas, stage
from etl.sources import BY_ID, SOURCES as REGISTRY, TABLES
from etl.sources import abs as abs_sources

pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 90)

def show(obj):
    """Display the source code of a function, so you can read exactly what runs."""
    display(Markdown("```python\n" + inspect.getsource(obj) + "\n```"))

info = {"Python": platform.python_version(), "Inside Docker": IN_DOCKER, "Project folder": str(ROOT),
        "pandas": pd.__version__}
try:
    import geopandas, pandera
    info.update(geopandas=geopandas.__version__, pandera=pandera.__version__)
except ImportError as e:
    info["MISSING LIBRARY"] = str(e)
display(pd.Series(info, name="value").to_frame())

## 2. How the Docker commands map to this code

The commands in `etl/README.md` all end up running the same Python package, `etl/`. Docker's only job is to give it
the right Python and libraries. **Your code is not copied into the Docker image**: `docker-compose.yml` mounts your
project folder into the container at `/work`, so the files that run are the ones you see and edit.

| You type | What happens | Where the code is |
|---|---|---|
| `docker compose build etl` | Builds an image: Python 3.11 plus the exact library versions in `requirements.lock`. Needed once (and when `requirements.lock` changes). | `Dockerfile`, `requirements.lock` |
| `docker compose run --rm etl` | Starts the database container `db`, then a container from that image with your project mounted at `/work`, and runs `bash scripts/etl.sh run`. `--rm` deletes the container afterwards; your files stay. | `docker-compose.yml` (service `etl`), `scripts/etl.sh` |
| *(inside the container)* `python -m etl run` | Python runs the package `etl`; its entry point reads the sub-command and calls `cmd_run`. | `etl/__main__.py`, `etl/cli.py` |
| `... etl run --credentials Credentials.json --db-host host.docker.internal --db-name postgres` | The same run, but loading into your PostgreSQL on port 5432. `host.docker.internal` is how a container reaches your computer. | `load.connection_settings` in `etl/load.py` |
| `... --entrypoint python etl download_data.py` | Downloads the sources into `raw/` and records them in `raw/manifest.json`. It is not part of the `etl/` package. | `download_data.py` |
| `... --entrypoint python etl -m pytest tests -q` | Runs the tests. | `tests/` |
| **this notebook** | Calls the same functions directly from Python. | `cli.cmd_run` and the modules below |

**What `cmd_run` does, for each source in order** (`etl/cli.py`):

1. `Snapshot.from_record(...)`: look the source up in `raw/manifest.json` (written by `download_data.py`) and check that
   each file is there and matches its checksum (`etl/core.py`). It returns a `Snapshot`: the files, their checksums and the
   release. A source that isn't downloaded yet is reported as `NOT RUN`.
2. `src.parse(snap)`: if the download is a zip, unzip it first (`snap.unzip()`, `extract_zip`), then read the files and
   return cleaned tables as pandas or GeoPandas frames. This is the code in `etl/sources/`.
3. `quality.assess(...)`: run the source's own checks (`src.check`) and the pandera schema for each table
   (`schemas.validate_table`). A FAIL stops that source here, so bad data is never written.
4. `stage.write_staged(...)` saves each table into `staging/`, and the manifest is updated with what
   was read (`raw/manifest.json`).
5. After all the sources: `load.load(...)` writes the tables into PostGIS (schema `vibrancy`), adds indexes and comments, and
   fills `vibrancy.etl_manifest` and `vibrancy.data_quality_results`.
6. It writes `staging/data_quality_report.csv` and `staging/run_report.json`, prints a summary, and returns 0 if
   everything worked.

**The files in `etl/`, and what each one does**

| File | What it holds |
|---|---|
| `cli.py` | The commands `run` and `list`, and `cmd_run`, the loop above. |
| `core.py` | Paths and constants; the `Manifest`; `Snapshot` (a source's downloaded files); `Source` (parse and check bundled together); `Ctx` (the run's manifest and quality records); `extract_zip`; small helpers such as `to_int`. |
| `stage.py` | Writes and reads the cleaned tables in `staging/` (GeoPackage for spatial tables, CSV for the rest). |
| `sources/abs.py` | ABS: SA2 boundaries, businesses, income, population, mesh block dwellings. |
| `sources/aec.py` | Australian Electoral Commission polling places. |
| `sources/nsw.py` | NSW school catchments and hospitals. |
| `sources/tfnsw.py` | Transport for NSW GTFS stops and traffic lights. |
| `sources/osm.py` | OpenStreetMap: amenities, crossings, walkable roads, daily-living shops. |
| `sources/cityofsydney.py` | City of Sydney trees, stairs and mobility parking. |
| `quality.py` | `QualityRecord` (PASS, WARNING or FAIL) and `assess`, which runs both layers of checks. |
| `schemas.py` | The pandera schema for each table: columns, types, allowed values, ranges. |
| `load.py` | Chooses the database and loads the tables, indexes, comments and the manifest into schema `vibrancy`. |

The next sections run these one at a time.

## 3. The sources

Each source is registered in `etl/sources/__init__.py`, in an order that matters (the SA2 boundaries come first because
other checks use them).

In [ ]:
registry = pd.DataFrame([{"source": s.id, "tables it makes": ", ".join(s.outputs), "what it is": s.title,
                          "licence": s.licence, "publisher": s.attribution} for s in REGISTRY])
display(registry)

## 4. One source, step by step

Every source is a `Source` with two functions: **parse** (clean into tables) and **check** (validate). Downloading is
done beforehand by `download_data.py`. `cmd_run` reads the manifest, then calls parse and check in order. Here we do
the same by hand for one small source, the ABS population estimates. The code for its `parse` is:

In [ ]:
show(abs_sources.parse_population)

It opens the workbook the manifest points at (`snap.path()`) and hands its worksheet to `parse_population_ws`.
**Step 1: read the manifest.** This looks up the file `download_data.py` saved and checks its checksum. It doesn't use the
network.

In [ ]:
ctx = core.Ctx()                       # the run's context: the manifest and the quality records
DEMO = BY_ID["abs_population"]
record = ctx.manifest.get(DEMO.id)     # written by download_data.py
snap = core.Snapshot.from_record(DEMO.id, record)   # raises an error if the file is missing or changed

print("release:", snap.release)
print("from   :", snap.url)
display(pd.DataFrame(snap.files).T[["path", "bytes", "sha256", "last_modified"]])

The `sha256` is a fingerprint of the file. It goes into the manifest so the ETL can prove it is using
exactly the file that was downloaded. **Step 2: parse.** The workbook has title rows above the data and one column per age band; the
parser finds the header row by its content, keeps the Greater Sydney rows, and renames the columns:

In [ ]:
show(abs_sources.parse_population_ws)

In [ ]:
tables = DEMO.parse(snap)              # {table name: DataFrame}
for name, df in tables.items():
    print(name, "->", df.shape[0], "rows x", df.shape[1], "columns")
    display(df.head(4))

**Step 3: check.** `quality.assess` runs the source's own checks (row count, every Greater Sydney SA2 present, plausible
totals) and the pandera schema for the table. Here is the code, then its result:

In [ ]:
show(quality.assess)

In [ ]:
records = quality.assess(DEMO, tables, ctx)
display(quality.records_to_frame(records))

Two records: one from the source's own checks (`abs_population`) and one from the pandera schema
(`pandera:population`). Any FAIL would stop this source. **Step 4: stage and record.** The table is saved into
`staging/` and the manifest is updated:

In [ ]:
for name, df in tables.items():
    print("wrote staging/" + stage.write_staged(name, df))
ctx.manifest.put(DEMO.id, snap.to_record())
ctx.manifest.save()
print("recorded in", core.MANIFEST_PATH.relative_to(ROOT))

## 5. The code that does the work

Four functions carry most of the weight. They are shown here so you can read them.

**Checking a file against the manifest** (`Snapshot.from_record`, `etl/core.py`). It rebuilds the source's record from
`raw/manifest.json` and refuses to go on if a file is missing (`NOT RUN`, with the command that downloads it) or its
SHA-256 no longer matches (the file changed since it was downloaded):

In [ ]:
show(core.Snapshot.from_record)

**Unzipping** (`extract_zip`, `etl/core.py`). Every zip a source delivers is unzipped into a folder next to it before
anything is read. The zip is kept (so its checksum can still be checked), an earlier extraction is reused, and a file
whose path would land outside the folder is refused:

In [ ]:
show(core.extract_zip)

**The pandera contract for a table** (`schemas.validate_table`). It looks up the table's schema and validates it,
collecting all the problems at once. A table with no problems returns PASS:

In [ ]:
show(schemas.validate_table)

**Choosing the database** (`load.connection_settings`, `etl/load.py`). By default it uses the settings docker compose
passes in; `--credentials`, `--db-host` and `--db-name` override them, which is how the data reaches your pgAdmin database:

In [ ]:
show(load.connection_settings)

And the loop that ties it together, `cmd_run`:

In [ ]:
show(cli.cmd_run)

## 6. Run the whole pipeline

This is the equivalent of `docker compose run --rm etl`. It reads every source in `SOURCES` (all of them by default) from `raw/`,
prints the same log, and returns 0 if everything worked. First, turn `LOAD_TO` into the options the command takes:

In [ ]:
import argparse

if LOAD_TO == "pgadmin":      # the PostgreSQL that pgAdmin shows, on port 5432 (database "postgres", not the "localhost" in Credentials.json)
    db = {"credentials": "Credentials.json", "db_host": "host.docker.internal" if IN_DOCKER else "localhost", "db_name": "postgres"}
else:                          # None (files only) or "compose" (the settings docker compose passes in)
    db = {"credentials": None, "db_host": None, "db_name": None}

args = argparse.Namespace(only=SOURCES, no_db=LOAD_TO is None, **db)
print("sources:", SOURCES or "all", "| load to:", LOAD_TO or "files only")

In [ ]:
exit_code = cli.cmd_run(args)
print("\nexit code:", exit_code, "(0 means every source was loaded)")

## 7. What was produced

**`raw/`** holds each download exactly as it arrived, in a dated folder per source, and, next to every zip, the folder it
was unzipped into:

In [ ]:
rows = []
for src_dir in sorted(core.RAW.glob("*")):
    if not src_dir.is_dir():
        continue
    for day in sorted(p for p in src_dir.iterdir() if p.is_dir()):
        for item in sorted(day.iterdir()):
            if item.name.endswith(".part"):
                continue
            size = sum(x.stat().st_size for x in item.rglob("*") if x.is_file()) if item.is_dir() else item.stat().st_size
            rows.append({"source": src_dir.name, "date": day.name, "item": item.name + ("/" if item.is_dir() else ""),
                         "MB": round(size / 1e6, 1), "kind": "unzipped folder" if item.is_dir() else "download"})
raw_files = pd.DataFrame(rows)
display(raw_files if len(raw_files) else "raw/ is empty")

**`raw/manifest.json`** records where each file came from, when, and its fingerprint:

In [ ]:
if core.MANIFEST_PATH.exists():
    manifest = json.loads(core.MANIFEST_PATH.read_text())
    display(pd.DataFrame([{"source": k, "release": v["release"], "retrieved (UTC)": v["retrieved_at"],
                           "files": ", ".join(Path(f["path"]).name for f in v["files"].values()), "from": v["url"][:70]}
                          for k, v in sorted(manifest["sources"].items())]))

**`staging/`** holds the cleaned tables: `.gpkg` files for tables with locations, `.csv` for the rest:

In [ ]:
rows = []
for src in REGISTRY:
    for table in src.outputs:
        df = stage.read_staged(table)
        if df is None:
            continue
        f = next(core.STAGING.glob(table + ".*"))
        rows.append({"table": table, "rows": len(df), "columns": df.shape[1], "file": f.name,
                     "MB": round(f.stat().st_size / 1e6, 1), "from source": src.id})
staged = pd.DataFrame(rows)
display(staged if len(staged) else "staging/ is empty")

**The quality report** has one line per check: the source's own checks and the pandera schema for each table. A WARNING is
something worth knowing (for example, a few traffic lights with typo coordinates that are kept with an empty location);
a FAIL would have stopped that source.

In [ ]:
report_path = core.STAGING / "data_quality_report.csv"
if report_path.exists():
    quality_report = pd.read_csv(report_path)
    colour = {"PASS": "#d4edda", "WARNING": "#fff3cd", "FAIL": "#f8d7da"}
    display(quality_report.style.map(lambda s: f"background-color: {colour.get(s, 'white')}", subset=["status"]))

## 8. Look at the data

In [ ]:
for table in ["sa2_boundaries", "businesses", "stops", "hospitals", "traffic_lights"]:
    df = stage.read_staged(table)
    if df is not None:
        print(f"{table}: {len(df):,} rows")
        display(df.drop(columns="geom", errors="ignore").head(3))

In [ ]:
import matplotlib.pyplot as plt

sa2 = stage.read_staged("sa2_boundaries")
if sa2 is None:
    print("Run the asgs_sa2 source (and traffic_lights, hospitals) to draw the map.")
else:
    fig, ax = plt.subplots(figsize=(8, 8))
    sa2.plot(ax=ax, color="#f0f0f0", edgecolor="#bbbbbb", linewidth=0.3)
    for name, colour, size in (("traffic_lights", "#1f77b4", 1), ("hospitals", "#d62728", 14)):
        points = stage.read_staged(name)
        if points is not None:
            points[points.geometry.notna()].plot(ax=ax, color=colour, markersize=size, label=name)
    minx, miny, maxx, maxy = sa2.total_bounds
    ax.set_xlim(minx, maxx); ax.set_ylim(miny, maxy)
    ax.set_title("Greater Sydney SA2 regions with traffic lights and hospitals")
    ax.legend(loc="lower left"); ax.set_axis_off()
    plt.show()

## 9. In the database

If `LOAD_TO` was set, the tables are in schema `vibrancy`. This lists them from the database itself:

In [ ]:
if LOAD_TO is None:
    print("LOAD_TO is None, so nothing was loaded. Set LOAD_TO = 'pgadmin' above and run section 6 again to load.")
else:
    from sqlalchemy import text

    if LOAD_TO == "pgadmin":
        settings = load.connection_settings(db["credentials"], db["db_host"], db["db_name"])
    else:
        settings = load.connection_settings()
    print("connected to", load.describe(settings))
    with load.engine(settings).connect() as conn:
        names = pd.read_sql(text("select table_name from information_schema.tables where table_schema = 'vibrancy' order by 1"),
                            conn)["table_name"].tolist()
        display(pd.DataFrame([{"table": n, "rows": conn.execute(text(f'select count(*) from vibrancy."{n}"')).scalar()} for n in names]))

## 10. Start over

To wipe what the ETL made and run again, set `RESET_FILES = True` in section 0 and run the next cell. It deletes `raw/`
and `staging/` only: never a database. To remove the `vibrancy` tables from a database, run
`DROP SCHEMA IF EXISTS vibrancy CASCADE;` in pgAdmin's Query Tool.

In [ ]:
if RESET_FILES:
    for folder in (core.RAW, core.STAGING):
        if folder.exists():
            shutil.rmtree(folder)
            print("deleted", folder.relative_to(ROOT))
else:
    print("RESET_FILES is False, so nothing was deleted.")